In [1]:
import os
os.chdir("..")

import pandas as pd
import git

from AugmentedNet.common import DATASPLITS, ANNOTATIONSCOREDUPLES


def resolve_dir(d):
    """Resolves '~' to HOME directory and turns ``d`` into an absolute path."""
    if d is None:
        return None
    d = str(d)
    if "~" in d:
        return os.path.expanduser(d)
    return os.path.abspath(d)

REPO = resolve_dir(".")
DATASET = resolve_dir("events")
SUBMODULES = resolve_dir("rawdata")

In [2]:
v100_ids = {}

i = 0
for split, files in DATASPLITS.items():
    for nickname in files:
        annotations_path, score_path = ANNOTATIONSCOREDUPLES[nickname]
        file_info = (nickname, split)
        # if annotations_path in v100_ids:
        #     print(f"{nickname} | anno: {annotations_path} was already in for {v100_ids[annotations_path]}")
        v100_ids[annotations_path] = file_info
        if score_path in v100_ids:
            existing_nn, existing_split = v100_ids[score_path]
            file_info = ((existing_nn, nickname), (existing_split, split))
            #print(f"{nickname} | score: {score_path} was already in for {v100_ids[score_path]}")
        v100_ids[score_path] = file_info
        i += 1
    
#assert len(v100_ids) == i * 2, f"dict length {len(v100_ids)} != {i * 2} ({i} * 2)"

In [3]:
repo = git.Repo(REPO)
augmentednet_version = "v1.0.0" 
submodule_commits = {
    sm.name: sm.hexsha
    for sm in repo.submodules
}
submodule_commits

{'TAVERN': '84dff27d34b9cc554822234b5aedbc59da43a7d2',
 'ABC': 'eb51669126f7870df210ff98a3a76ea07581aee8',
 'haydn_op20_harm': '264bfa10eca442c0b3d7d4e1907d782b80b1599d',
 'When-in-Rome': 'f45b37c85cec7ee5c9b205bb5c8a64f80cf2f821',
 'music21_corpus': 'ed021ae6804fd6af3e9723b502094170be7773ab',
 'functional-harmony-micchi': 'b401ddfea6c337b21d0a7681337c6058e5411dc3'}

In [4]:
data = []
for data_dir in os.listdir(SUBMODULES):
    if data_dir in submodule_commits:
        repo_name = data_dir
        version = submodule_commits[data_dir]
    else:
        repo_name = "AugmentedNet"
        version = augmentednet_version
    version = submodule_commits.get(data_dir, augmentednet_version)
    for path, subdirs, files in os.walk(os.path.join(SUBMODULES, data_dir)):
        rel_path = os.path.relpath(path, REPO)
        if rel_path == os.path.join("rawdata", "When-in-Rome"):
            subdirs[:] = ["Corpus"]
            continue
        for file in files:
            fname, fext = os.path.splitext(file)
            if not fext or fext in (".md", ".csv", ".tsv", ".py", ".sh", ".pdf"):
                continue
            fname_lower = fname.lower()
            if "feedback" in fname_lower or "template" in fname_lower:
                continue
            filepath = os.path.join(rel_path, file)
            info_dict = dict(path=filepath, extension=fext[1:])
            if filepath in v100_ids:
                nickname, split = v100_ids[filepath]
                info_dict["v1.0.0_id"] = nickname
                info_dict["v1.0.0_split"] = split
            data.append(info_dict)
            
df = pd.DataFrame.from_records(data)      
df.to_csv("augnet_rawdata_overview.tsv", sep="\t", index=False)
df.head()

,path,extension,v1.0.0_id,v1.0.0_split
0,rawdata/key_modulation_dataset/requirements.txt,txt,NaN,NaN
1,rawdata/key_modulation_dataset/reger/97B.krn,krn,NaN,NaN
2,rawdata/key_modulation_dataset/reger/87A.krn,krn,NaN,NaN
3,rawdata/key_modulation_dataset/reger/95B.krn,krn,NaN,NaN
4,rawdata/key_modulation_dataset/reger/41.krn,krn,NaN,NaN


In [5]:
len(v100_ids)

679

In [6]:
path_df = df.set_index("path")
for filepath, (nickname, split) in v100_ids.items():
    path_df.loc[filepath]
    # info_dict["v1.0.0_id"] = nickname
    # info_dict["v1.0.0_split"] = split

In [7]:
summary = pd.read_csv(
    os.path.join(DATASET, "dataset_summary.tsv"),
    sep="\t",
    index_col=0
)
summary

,file,annotation,score,collection,split,a_composer,a_title,s_movementName,s_composer,s_title,...,s_copyright,s_electronic encoder,s_movementNumber,s_number,s_opusNumber,s_groupTitle,s_lyricist,s_arranger,s_translator,s_None
0,bps-01-op002-no1-1,rawdata/When-in-Rome/Corpus/Piano_Sonatas/Beet...,rawdata/functional-harmony-micchi/data/BPS/sco...,bps,test,Beethoven,"Piano Sonata 1, Op.2 No.1, Movement 1",bps_01_01.mxl,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,bps-14-op027-no2-moonlight-1,rawdata/corrections/WiR/Corpus/Piano_Sonatas/B...,rawdata/functional-harmony-micchi/data/BPS/sco...,bps,test,Beethoven,"Piano Sonata 14, Op.27 No.2(Moonlight), Moveme...",bps_14_01.mxl,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,bps-23-op057-appassionata-1,rawdata/corrections/WiR/Corpus/Piano_Sonatas/B...,rawdata/functional-harmony-micchi/data/BPS/sco...,bps,test,Beethoven,"Piano Sonata 23, Op.57(Appassionata), Movement 1",bps_23_01.mxl,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,bps-15-op028-pastorale-1,rawdata/corrections/WiR/Corpus/Piano_Sonatas/B...,rawdata/functional-harmony-micchi/data/BPS/sco...,bps,test,Beethoven,"Piano Sonata 15, Op.28(Pastorale), Movement 1",bps_15_01.mxl,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,bps-10-op014-no2-1,rawdata/corrections/WiR/Corpus/Piano_Sonatas/B...,rawdata/functional-harmony-micchi/data/BPS/sco...,bps,test,Beethoven,"Piano Sonata 10, Op.14 No.2, Movement 1",bps_10_01.mxl,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
348,wir-monteverdi-madrigals-book-3-1,rawdata/When-in-Rome/Corpus/Early_Choral/Monte...,rawdata/When-in-Rome/Corpus/Early_Choral/Monte...,wir,training,Claudio Monteverdi,La Giovinetta Pianta,CANTO,1. LA GIOVINETTA PIANTA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
349,wir-variations-and-grounds-bach-b-minor-mass-b...,rawdata/When-in-Rome/Corpus/Variations_and_Gro...,rawdata/When-in-Rome/Corpus/Variations_and_Gro...,wir,training,Bach,"Crucifixus (from the B Minor mass, BWV232)","Bach Crucifixus (from the B Minor mass, BWV232)",NaN,NaN,...,CC0 1.0 Universal (Public Domain). Mark Gotham...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
350,wir-variations-and-grounds-purcell-sonata-z807,rawdata/When-in-Rome/Corpus/Variations_and_Gro...,rawdata/When-in-Rome/Corpus/Variations_and_Gro...,wir,training,Purcell,Purcell Sonata in G Minor Z807,Purcell Sonata in G Minor (Z 807),NaN,NaN,...,CC0 1.0 Universal (Public Domain). Mark Gotham...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
351,wir-variations-and-grounds-purcell-chacony-z730,rawdata/When-in-Rome/Corpus/Variations_and_Gro...,rawdata/When-in-Rome/Corpus/Variations_and_Gro...,wir,training,Purcell,Chacony (Chaconne) in G Minor Z730,score.mxl,Purcell,Chacony (Chaconne),...,CC0 1.0 Universal (Public Domain). Mark Gotham...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
